<a href="https://colab.research.google.com/github/esalinasbio/LCG_StructuralBio_Dynamics/blob/main/notebooks/Dinamica_Conformacional_AdK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 3: Dinámica de proteínas: fundamentos

Introducción a la Biología estructural, 10 de noviembre del 2026

---

### El sistema: adenilato cinasa de *E. coli* (AdK)

La AdK cataliza la reacción:

$$\text{ATP} + \text{AMP} \rightleftharpoons 2\,\text{ADP}$$

Tiene tres dominios: un **CORE** rígido, un dominio **LID** que se cierra sobre el ATP y un dominio **NMP** que se cierra sobre el AMP. Para la catálisis, la proteína tiene que **cerrarse**; para liberar el producto, tiene que **abrirse** otra vez. En AdK, la apertura es uno de los pasos más lentos del ciclo catalítico: la función depende directamente de un cambio conformacional.

Hay dos estructuras clásicas:

- **4AKE**: AdK sin ligando, **abierta**.
- **1AKE**: AdK con un inhibidor (Ap5A) que imita a los dos sustratos, **cerrada**.

---

### Qué vamos a hacer

| Parte | Qué hacemos | Idea |
|---|---|---|
| **A** | Analizar el cambio conformacional | La estructura es un ensamble, cómo es el cambio confomracional |
| A1 | Abierta vs cerrada | ¿Cuánto cambia y qué se mueve? |
| A2 | Ángulos diedros | ¿Dónde está la bisagra? |
| A3 | Interpolación | La dirección del movimiento |
| A4 | Todas las AdK del PDB + AlphaFoldDB | El PDB contiene múltiples estados, una predicción da una sola estructura |
| A5 | Factores B y un modelo elástico | ¿La estructura de la proteína determina cómo se mueve? |
| **B** | Preparar una simulación | Protonar, campo de fuerza, caja, minimizar, equilibrar, producir |
| **C** | ¿Qué cambió en 200 ps? | Las escalas de tiempo: qué le alcanza a la MD y que no |

---

---
## Paso 0: Entorno

* **`Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `T4 GPU`**. Si ya esta así, todo bien.

* Presiona `Ctrl + S` o usa el menú `Archivo` (`File`) para guardar una copia del notebook en tu Google Drive.

Primero hayq ue ejecutar las celdas de instalación:

- **0a**: instala las herramientas de análisis. Para este ejercicio, utilizaremos `mdtraj` para procesar y analizar las estructuras/trayectorias y `py3Dmol` para visualizarlas. Con eso alcanza para toda la Parte A.
- **0b**: instala `OpenMM` y `AmberTools` **en segundo plano**. `OpenMM` es el programa de dinámica molecualr que utilizaremos en el ejercicio, `AmberTools` nos ayudará a procesar las trayectorias generadas. Mientras se instala, sigan con la Parte A. En la parte B empieza comprobando que ya terminó.

In [ ]:
#@title 0a: Herramientas de análisis
import time; _t0 = time.time()
!pip install -q mdtraj py3Dmol
import os, sys, json, subprocess, urllib.request
import numpy as np, matplotlib.pyplot as plt, mdtraj as mdt, py3Dmol
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

C_PROT, C_ACENTO, C_AGUA, C_ION, GRIS = "#2E8B8B", "#E8813A", "#9fc7d8", "#c8a02c", "#64748b"
MORADO, NARANJA, ROJO = "#7c3aed", "#E8813A", "#b91c1c"

# dominios de AdK (numeración de E. coli, 1-214)
N_RES = 214
DOMINIOS = {"CORE": [(1, 29), (60, 121), (160, 214)], "NMP": [(30, 59)], "LID": [(122, 159)]}
C_DOM = {"CORE": C_PROT, "NMP": MORADO, "LID": NARANJA}
RES = np.arange(1, N_RES + 1)
MASK = {k: np.isin(RES, [r for a, b in v for r in range(a, b + 1)]) for k, v in DOMINIOS.items()}
AA = ["ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS","MET","PHE",
      "PRO","SER","THR","TRP","TYR","VAL","HID","HIE","HIP","HSD","HSE","HSP","CYX","ASH","GLH","LYN"]
_lista = lambda dom: [r for a, b in DOMINIOS[dom] for r in range(a, b + 1)]
ESTILO_DOMINIOS = [({"resn": AA}, {"cartoon": {"color": C_DOM["CORE"]}}),
                   ({"resn": AA, "resi": _lista("NMP")}, {"cartoon": {"color": C_DOM["NMP"]}}),
                   ({"resn": AA, "resi": _lista("LID")}, {"cartoon": {"color": C_DOM["LID"]}})]

def sombrear_dominios(ax, alpha=0.10):
    for dom, rangos in DOMINIOS.items():
        for a, b in rangos:
            ax.axvspan(a - 0.5, b + 0.5, color=C_DOM[dom], alpha=alpha, lw=0)
    for dom in ("CORE", "NMP", "LID"):
        ax.plot([], [], color=C_DOM[dom], lw=6, alpha=0.4, label=dom)

# visualización
def _aplicar(v, estilos, viewer=None):
    for k, (sel, est) in enumerate(estilos):
        kw = {} if viewer is None else {"viewer": viewer}
        (v.setStyle if k == 0 else v.addStyle)(sel, est, **kw)

def ver(archivo, estilos, ancho=780, alto=430, girar=False, celda=False):
    v = py3Dmol.view(width=ancho, height=alto)
    v.addModel(open(archivo).read(), "pdb")
    _aplicar(v, estilos)
    if celda: v.addUnitCell({"box": {"color": "#94a3b8"}})
    v.zoomTo()
    if girar: v.spin(True)
    v.show()

def ver_animacion(archivo_multi, estilos, ancho=780, alto=440, intervalo=150, bucle="forward"):
    v = py3Dmol.view(width=ancho, height=alto)
    v.addModelsAsFrames(open(archivo_multi).read(), "pdb")
    _aplicar(v, estilos)
    v.zoomTo(); v.animate({"loop": bucle, "interval": intervalo}); v.show()

# estructuras
def descargar(pdb_id, carpeta="pdb"):
    """descarga un PDB del RSCB-PDB"""
    os.makedirs(carpeta, exist_ok=True)
    ruta = f"{carpeta}/{pdb_id.upper()}.pdb"
    if not os.path.exists(ruta):
        try:
            urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb", ruta)
        except Exception:
            return None
    return ruta

def cadenas_adk(ruta, min_ca=190):
    t = mdt.load_pdb(ruta)[0]
    out = []
    for ch in t.topology.chains:
        ca = np.full((N_RES, 3), np.nan); seq = {}
        for a in ch.atoms:
            r = a.residue
            if a.name == "CA" and r.is_protein and 1 <= r.resSeq <= N_RES:
                ca[r.resSeq - 1] = 10 * t.xyz[0, a.index]; seq[r.resSeq] = r.code
        if np.isfinite(ca[:, 0]).sum() >= min_ca:
            out.append({"cadena": getattr(ch, "chain_id", None) or "ABCDEFGH"[ch.index % 8],
                        "indice": ch.index, "ca": ca, "seq": seq})
    return t, out

def bfactores_ca(ruta, cadena):
    b = np.full(N_RES, np.nan)
    for l in open(ruta):
        if l.startswith("ATOM") and l[12:16].strip() == "CA" and l[21] == cadena and l[16] in " A":
            r = int(l[22:26])
            if 1 <= r <= N_RES: b[r - 1] = float(l[60:66])
    return b

def kabsch(P, Q):
    pc, qc = P.mean(0), Q.mean(0)
    U, S, Vt = np.linalg.svd((P - pc).T @ (Q - qc))
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = U @ np.diag([1, 1, d]) @ Vt
    return R, qc - pc @ R

def alinear(ca_mov, ca_ref, mascara=None):
    m = np.ones(N_RES, bool) if mascara is None else mascara
    ok = m & np.isfinite(ca_mov[:, 0]) & np.isfinite(ca_ref[:, 0])
    R, t = kabsch(ca_mov[ok], ca_ref[ok])
    return ca_mov @ R + t, (R, t)

def rmsd(a, b, mascara=None):
    m = np.ones(N_RES, bool) if mascara is None else mascara
    ok = m & np.isfinite(a[:, 0]) & np.isfinite(b[:, 0])
    return np.sqrt(((a[ok] - b[ok]) ** 2).sum(1).mean())

def distancias_dominio(ca):
    """resultado en Angstroms"""
    c = {k: np.nanmean(ca[..., m, :], axis=-2) for k, m in MASK.items()}
    return (np.linalg.norm(c["NMP"] - c["CORE"], axis=-1),
            np.linalg.norm(c["LID"] - c["CORE"], axis=-1))

print(f"Listo: {time.time()-_t0:.0f} s")

In [ ]:
#@title 0b: Instalar OpenMM y AmberTools en segundo plano
#@markdown Pueden seguir con la Parte A mientras termina.
_pasos = f"""
wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
bash Miniforge3-Linux-x86_64.sh -bfp /usr/local
mamba config --set auto_update_conda false
{sys.executable} -m pip install -q "openmm[cuda12]" pdbfixer
conda install -y -q -c conda-forge ambertools
echo INSTALACION_TERMINADA
"""
if "INSTALACION" not in globals() or INSTALACION.poll() is not None:
    INSTALACION = subprocess.Popen(["bash", "-c", _pasos], stdout=open("instalacion.log", "w"),
                                   stderr=subprocess.STDOUT)
print("Instalación corriendo en segundo plano. Sigan con la Parte A.")

---
# Parte A: Analizar el cambio conformacional

Antes de correr una simulación, ya hay mucha información sobre la dinámica en las estructuras experimentales, vamos a verla.

## A1: Abierta vs cerrada

Descargamos 4AKE (abierta) y 1AKE (cerrada), tomamos la cadena A de cada una y las superponemos **sobre el dominio CORE**. Así, lo que veamos moverse es el movimiento de LID y NMP *relativo* al CORE.

> ### Pregunta:
> **¿Cuánto RMSD hay entre la abierta y la cerrada: 1, 5 o 15 Å?**
>
> **¿Qué dominio se mueve más?**

In [ ]:
#@title A1: Superponer 4AKE y 1AKE sobre el CORE
t_ab, c_ab = cadenas_adk(descargar("4AKE"))
t_ce, c_ce = cadenas_adk(descargar("1AKE"))
ca_ab, ca_ce = c_ab[0]["ca"], c_ce[0]["ca"] # cadena A de cada una

ca_ce_core, (R_core, t_core) = alinear(ca_ce, ca_ab, MASK["CORE"])
ca_ce_glob, _                = alinear(ca_ce, ca_ab)
desp = np.linalg.norm(ca_ce_core - ca_ab, axis=1)

print("=" * 60)
print(f"  RMSD Cα global (alineando todo)      : {rmsd(ca_ce_glob, ca_ab):6.2f} Å")
print(f"  RMSD Cα del CORE (alineando el CORE) : {rmsd(ca_ce_core, ca_ab, MASK['CORE']):6.2f} Å")
print("-" * 60)
print("  Cuánto viaja en promedio cada Cα de abierta a cerrada:")
for dom in ("CORE", "NMP", "LID"):
    print(f"     {dom:<4s}                              : {np.nanmean(desp[MASK[dom]]):6.2f} Å")
print("=" * 60)
dN_ab, dL_ab = distancias_dominio(ca_ab); dN_ce, dL_ce = distancias_dominio(ca_ce)
print(f"  Distancia LID–CORE : abierta {dL_ab:5.1f} Å, cerrada {dL_ce:5.1f} Å")
print(f"  Distancia NMP–CORE : abierta {dN_ab:5.1f} Å, cerrada {dN_ce:5.1f} Å")

fig, ax = plt.subplots(figsize=(11, 3.4))
sombrear_dominios(ax)
ax.plot(RES, desp, color="k", lw=1.3)
ax.set_xlabel("residuo"); ax.set_ylabel("desplazamiento del Cα (Å)")
ax.set_title("¿Cuánto se mueve cada residuo de abierta a cerrada?")
ax.set_xlim(1, N_RES); ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

# estructuras completas (cadena A) para ver en 3D
def cadena_completa(t, info):
    return t.atom_slice(t.topology.select(f"chainid {info['indice']} and protein"))
ab_A = cadena_completa(t_ab, c_ab[0]); ab_A.save_pdb("4AKE_A.pdb")
ce_A = cadena_completa(t_ce, c_ce[0])
ce_A.xyz = ((10 * ce_A.xyz) @ R_core + t_core) / 10 # misma superposición
ce_A.save_pdb("1AKE_A_alineada.pdb")

b_por_atomo = np.array([np.nan_to_num(desp[a.residue.resSeq - 1]) if 1 <= a.residue.resSeq <= N_RES else 0
                        for a in ab_A.topology.atoms])
ab_A.save_pdb("4AKE_A_desplazamiento.pdb", bfactors=np.clip(b_por_atomo, 0, 99))

v = py3Dmol.view(width=880, height=430, viewergrid=(1, 2))
v.addModel(open("4AKE_A_desplazamiento.pdb").read(), "pdb", viewer=(0, 0))
v.setStyle({}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb",
                                            "min": float(np.nanmax(desp)), "max": 0}}}, viewer=(0, 0))
v.addModel(open("1AKE_A_alineada.pdb").read(), "pdb", viewer=(0, 1))
_aplicar(v, ESTILO_DOMINIOS, viewer=(0, 1))
v.addModel(open("4AKE_A.pdb").read(), "pdb", viewer=(0, 1))
v.setStyle({"model": 1}, {"cartoon": {"color": "#94a3b8", "opacity": 0.45}}, viewer=(0, 1))
v.zoomTo(); v.show()
print("Izquierda: 4AKE coloreada por desplazamiento (rojo = se mueve mucho, azul = casi nada).")
print("Derecha: 1AKE (cerrada) por dominio (CORE verde-azul, NMP morado, LID naranja) y 4AKE (abierta) en gris.")

El **desplazamiento** de un residuo es la distancia que recorre su Cα entre la estructura abierta y la cerrada.

El RMSD global mezcla dos cosas: el CORE casi no cambia y LID y NMP se mueven decenas de Å.

## A2: ¿Dónde está la bisagra?

Si LID y NMP se mueven como bloques, los residuos dentro casi no cambian su estructura local. El cambio conformacional tiene que concentrarse en unos pocos residuos que funcionan como **bisagras**.

Los ángulos diedros del backbone (φ, ψ) describen la geometría local, y son independientes de cómo superpongamos las estructuras.

> ### Pregunta
> **¿Cuántos residuos creen que cambian sus ángulos de forma importante?**

In [ ]:
#@title A2: Cambios en φ y ψ entre abierta y cerrada
UMBRAL_GRADOS = 60   #@param {type:"number"}

def diedros(t):
    out = {}
    for nombre, f, col in (("phi", mdt.compute_phi, 1), ("psi", mdt.compute_psi, 0)):
        idx, ang = f(t)
        out[nombre] = {t.topology.atom(i[col]).residue.resSeq: np.degrees(a)
                       for i, a in zip(idx, ang[0])}
    return out

d_ab, d_ce = diedros(ab_A), diedros(cadena_completa(t_ce, c_ce[0]))
cambio = np.zeros(N_RES)
for ang in ("phi", "psi"):
    for r, a in d_ab[ang].items():
        if r in d_ce[ang] and 1 <= r <= N_RES:
            delta = abs((a - d_ce[ang][r] + 180) % 360 - 180)
            cambio[r - 1] = max(cambio[r - 1], delta)

FRONTERAS = [29.5, 59.5, 121.5, 159.5]   # límites entre dominios
fig, ax = plt.subplots(figsize=(11, 3.6))
sombrear_dominios(ax)
ax.bar(RES, cambio, color="k", width=0.9, label="cambio de diedros")
ax.axhline(UMBRAL_GRADOS, ls="--", c=ROJO, lw=1.2)
for x in FRONTERAS: ax.axvline(x, ls=":", c=GRIS, lw=1)
ax.set_xlabel("residuo"); ax.set_ylabel("máx |Δφ|, |Δψ| (°)"); ax.set_xlim(1, N_RES)
ax2 = ax.twinx()
ax2.plot(RES, desp, color=GRIS, lw=1.6, alpha=0.8, label="desplazamiento (A1)")
ax2.set_ylabel("desplazamiento del Cα (Å)", color=GRIS)
ax.set_title("Cambio local (diedros) contra movimiento global (desplazamiento)")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left", ncol=5); ax.grid(alpha=0.25, axis="y")
plt.tight_layout(); plt.show()

grandes = RES[cambio > UMBRAL_GRADOS]
dist_frontera = lambda r: min(abs(r - f) for f in FRONTERAS)
print(f"Residuos con cambio > {UMBRAL_GRADOS}°: {len(grandes)} de {N_RES}\n")
print(f"{'residuo':>8}{'dominio':>8}{'Δ diedro':>10}{'a la frontera más cercana':>28}")
for r in grandes:
    dom = next(k for k, m in MASK.items() if m[r - 1])
    print(f"{r:>8d}{dom:>8s}{cambio[r-1]:>9.0f}°{dist_frontera(r):>20.1f} residuos")

v = py3Dmol.view(width=780, height=430)
v.addModel(open("4AKE_A.pdb").read(), "pdb")
_aplicar(v, ESTILO_DOMINIOS)
v.addStyle({"resi": [int(r) for r in grandes]}, {"stick": {"radius": 0.25, "colorscheme": "redCarbon"}})
v.zoomTo(); v.show()
print("En rojo (sticks): los residuos que más cambian sus diedros.")

Cómo leerlo: la curva gris es casi cero en el CORE y crece en LID y NMP, más entre más lejos del eje de giro (como una puerta, el borde se mueve más que la parte junto a las bisagras). Un residuo que cambia mucho sus diedros está actuando como bisagra. Uno que cambia dentro de cada dominio, es un reacomodo local.

Este análisis tiene un límite, solo detecta cambios bruscos en un residuo (por arriba del umbral). Una bisagra también puede tener un giro moderado entre varios residuos. Hay programas dedicados (por ejemplo, `DynDom`) que identifican los dominios rígidos y el eje de giro a partir de las dos estructuras.

> ### Pregunta
> De los residuos de la tabla, ¿cuáles son bisagras y cuáles reacomodos internos? Prueben bajar el umbral a 30°: ¿qué aparece?

## A3: La dirección del movimiento

Una interpolación lineal entre las dos estructuras **no es una trayectoria**: no tiene tiempo, ni energía, ni garantiza que la geometría intermedia sea físicamente posible. Sirve sólo para ver la *dirección* del cambio.

In [ ]:
#@title A3: Interpolar de abierta a cerrada
N_CUADROS = 24
clave = lambda a: (a.residue.resSeq, a.name)
pos_ce = {clave(a): ce_A.xyz[0, a.index] for a in ce_A.topology.atoms}
comunes = [a.index for a in ab_A.topology.atoms if clave(a) in pos_ce]
sub = ab_A.atom_slice(comunes)
xa = sub.xyz[0]
xc = np.array([pos_ce[clave(a)] for a in sub.topology.atoms])
marcos = np.array([(1 - s) * xa + s * xc for s in np.linspace(0, 1, N_CUADROS)])
mdt.Trajectory(marcos, sub.topology).save_pdb("interpolacion_4AKE_1AKE.pdb")

ver_animacion("interpolacion_4AKE_1AKE.pdb", ESTILO_DOMINIOS, intervalo=90, bucle="backAndForth")

## A4: El PDB contiene múltiples estados

4AKE y 1AKE no son las únicas estructuras de AdK de *E. coli*: hay decenas, con y sin ligandos, mutantes, en diferentes condiciones. Cada una es una estructura de la proteína en un estado.

Vamos a ponerlas todas en un mismo mapa de dos coordenadas que describen la función:

- **NMP–CORE**: distancia entre los centros de esos dos dominios.
- **LID–CORE**: lo mismo para el LID.

Y agregamos la predicción de **AlphaFold** (AlphaFoldDB) para la misma secuencia.

> ### Pregunta
> **¿Las estructuras caerán en dos grupos (abiertas y cerradas) o habrá intermedias?**
>
> **¿AlphaFold predecirá la forma abierta, la cerrada, o algo intermedio?**

In [ ]:
#@title A4: Todas las AdK de E. coli del PDB + AlphaFold
UNIPROT = "P69441" # AdK de E. coli
PDB_ADK = ["1AKE", "1ANK", "1E4V", "1E4Y", "2ECK", "3HPQ", "3HPR", "4AKE", "4X8H", "4X8L",
           "4X8M", "4X8O", "5EJE", "6F7U", "6HAM", "6HAP", "6RZE", "6S36", "7APU", "8BQF",
           "8CRG", "8PVW", "8Q2B", "8RJ4", "8RJ6", "8RJ9", "9L14", "9R6U", "9R71"]
NO_LIGANDO = {"HOH", "SO4", "PO4", "GOL", "EDO", "PEG", "PGE", "CL", "NA", "MG", "ZN", "CA",
              "K", "ACT", "FMT", "MPD", "TRS", "EPE", "DMS", "BME", "NI", "CO", "MN", "IMD"}

def tiene_ligando(t):
    for r in t.topology.residues:
        if not r.is_protein and not r.is_water and r.name not in NO_LIGANDO:
            if sum(1 for a in r.atoms if a.element.symbol != "H") >= 15:
                return True
    return False

seq_ref = c_ab[0]["seq"]
filas, omitidas = [], []
for pid in PDB_ADK:
    ruta = descargar(pid)
    if ruta is None:
        omitidas.append(f"{pid} (no se pudo bajar en .pdb)"); continue
    try:
        t, cads = cadenas_adk(ruta)
    except Exception as e:
        omitidas.append(f"{pid} ({type(e).__name__})"); continue
    lig = tiene_ligando(t)
    for c in cads:
        comunes = [r for r in c["seq"] if r in seq_ref]
        ident = np.mean([c["seq"][r] == seq_ref[r] for r in comunes]) if comunes else 0
        if ident < 0.9:
            continue
        ca_al, _ = alinear(c["ca"], ca_ab, MASK["CORE"])
        dN, dL = distancias_dominio(c["ca"])
        filas.append({"id": f"{pid}_{c['cadena']}", "pdb": pid, "ligando": lig,
                      "dN": dN, "dL": dL, "ca": ca_al})

# AlphaFold DB: la API da la URL del modelo vigente
def bajar_alphafold(uniprot, ruta="AF_modelo.pdb"):
    try:
        with urllib.request.urlopen(f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot}") as r:
            url = json.load(r)[0]["pdbUrl"]
        urllib.request.urlretrieve(url, ruta); return ruta
    except Exception:
        for ver_ in range(6, 0, -1):
            try:
                urllib.request.urlretrieve(
                    f"https://alphafold.ebi.ac.uk/files/AF-{uniprot}-F1-model_v{ver_}.pdb", ruta)
                return ruta
            except Exception:
                pass
    return None

ruta_af = bajar_alphafold(UNIPROT)
af = None
if ruta_af:
    _, c_af = cadenas_adk(ruta_af)
    ca_af = c_af[0]["ca"]
    plddt = np.nanmean(bfactores_ca(ruta_af, "A"))
    dN_af, dL_af = distancias_dominio(ca_af)
    af = {"dN": dN_af, "dL": dL_af, "ca": alinear(ca_af, ca_ab, MASK["CORE"])[0], "plddt": plddt}

print(f"Cadenas de AdK analizadas: {len(filas)} (de {len(set(f['pdb'] for f in filas))} entradas)")
if omitidas: print("Omitidas:", ", ".join(omitidas))
print(f"Con ligando: {sum(f['ligando'] for f in filas)} · sin ligando: {sum(not f['ligando'] for f in filas)}")
if af: print(f"AlphaFold: NMP–CORE {af['dN']:.1f} Å · LID–CORE {af['dL']:.1f} Å · pLDDT medio {af['plddt']:.0f}")

# PCA de todas las estructuras, Cα xyz
pila = np.array([f["ca"] for f in filas])
comun = np.isfinite(pila[:, :, 0]).all(axis=0)
Xp = pila[:, comun].reshape(len(filas), -1)
media = Xp.mean(0)
_, S, Vt = np.linalg.svd(Xp - media, full_matrices=False)
var_pca = 100 * S**2 / (S**2).sum()
proy = (Xp - media) @ Vt[:2].T
proy_af = ((af["ca"][comun].ravel() - media) @ Vt[:2].T) if af else None

# RMSD de cada estructura contra la abierta (4AKE) y la cerrada (1AKE)
rmsd_vs = lambda ca, ref: rmsd(alinear(ca, ref)[0], ref)
for f in filas:
    f["r_ab"], f["r_ce"] = rmsd_vs(f["ca"], ca_ab), rmsd_vs(f["ca"], ca_ce)
if af:
    af["r_ab"], af["r_ce"] = rmsd_vs(af["ca"], ca_ab), rmsd_vs(af["ca"], ca_ce)

fig, ax = plt.subplots(1, 3, figsize=(18, 5.2))
for a, (x, y) in zip(ax, [([f["dN"] for f in filas], [f["dL"] for f in filas]),
                          ([f["r_ce"] for f in filas], [f["r_ab"] for f in filas]),
                          (proy[:, 0], proy[:, 1])]):
    col = [NARANJA if f["ligando"] else C_PROT for f in filas]
    a.scatter(x, y, c=col, s=55, edgecolor="k", lw=0.5, zorder=3)
    for f, xi, yi in zip(filas, x, y):
        if f["pdb"] in ("4AKE", "1AKE") and f["id"].endswith("_A"):
            a.annotate(f["pdb"], (xi, yi), xytext=(6, 6), textcoords="offset points", fontsize=9, weight="bold")
    a.scatter([], [], c=C_PROT, edgecolor="k", label="sin ligando")
    a.scatter([], [], c=NARANJA, edgecolor="k", label="con ligando")
    a.grid(alpha=0.25)
if af:
    ax[0].scatter(af["dN"], af["dL"], marker="*", s=380, c="gold", edgecolor="k", zorder=4, label="AlphaFold")
    ax[1].scatter(af["r_ce"], af["r_ab"], marker="*", s=380, c="gold", edgecolor="k", zorder=4, label="AlphaFold")
    ax[2].scatter(*proy_af, marker="*", s=380, c="gold", edgecolor="k", zorder=4, label="AlphaFold")
ax[0].set_xlabel("NMP–CORE (Å)"); ax[0].set_ylabel("LID–CORE (Å)")
ax[0].set_title("Distancia entre dominios"); ax[0].legend(fontsize=8)
lim = 1.08 * max(max(f["r_ab"] for f in filas), max(f["r_ce"] for f in filas))
ax[1].plot([0, lim], [0, lim], ls="--", c=GRIS, lw=1, zorder=1)
ax[1].text(0.5 * lim, 0.53 * lim, "igual de parecida a ambas", rotation=45, rotation_mode="anchor",
           ha="center", fontsize=8, color=GRIS)
ax[1].set_xlim(-0.04 * lim, lim); ax[1].set_ylim(-0.04 * lim, lim); ax[1].set_aspect("equal")
ax[1].set_xlabel("RMSD Cα contra 1AKE, cerrada (Å)"); ax[1].set_ylabel("RMSD Cα contra 4AKE, abierta (Å)")
ax[1].set_title("¿A cuál se parece más?"); ax[1].legend(fontsize=8, loc="upper right")
ax[2].set_xlabel(f"PC1 ({var_pca[0]:.0f} %)"); ax[2].set_ylabel(f"PC2 ({var_pca[1]:.0f} %)")
ax[2].set_title("PCA"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"La PC1 sola explica {var_pca[0]:.0f} % de las diferencias entre todas las estructuras.")

Cómo leerlo:

- Las estructuras **con ligando** tienden a estar cerradas y **sin ligando** abiertas. Pero ¿hay alguna sin ligando cerrada, o intermedia? Si la proteína apo también visita conformaciones cerradas, el ligando **selecciona** un estado que ya existía (selección conformacional) en vez de **inducirlo** (ajuste inducido).
- Un solo movimiento colectivo domina las diferencias.
- **AlphaFold** da *una* estructura, sin decir qué tan probable es frente a las otras. El PDB da *varias*, pero cada cristal es un estado atrapado en condiciones particulares. **Ninguno de los dos dice cuánto tiempo pasa la proteína en cada región del mapa.** Eso es lo que buscamos con la dinámica molecular.

> ### Pregunta
> ¿Qué haría falta para saber si AdK sin ligando pasa el 1 %, el 10 % o el 50 % del tiempo cerrada?

## A5: Flexibilidad sin dinámica

Dos fuentes de información sobre flexibilidad sin tener que simular:

1. **Factores B** del cristal: cuantifica la dispersión de la densidad electrónica de cada átomo. Mezclan movimiento real con desorden del cristal, pero dan una idea de qué partes son más móviles.
2. **Un modelo de red elástica**: cada Cα es una esfera, y todas las que están a menos de 15 Å se unen con resortes idénticos. Nada de química, solo con la estructura de la proteína. Los modos de vibración más lentos de esa red son los movimientos colectivos más "fáciles".

> ### Pregunta
> **¿Puede un modelo de resortes, sin campo de fuerza ni agua, predecir hacia dónde se cierra la AdK?**

In [ ]:
#@title A5: Factores B y modelo de red elástica sobre 4AKE
CORTE_ANM = 15.0   #@param {type:"number"}

ok = np.isfinite(ca_ab[:, 0])
X = ca_ab[ok]; n = len(X)
H = np.zeros((3 * n, 3 * n))
for i in range(n):
    for j in range(i + 1, n):
        d = X[j] - X[i]; r2 = d @ d
        if r2 < CORTE_ANM**2:
            bloque = -np.outer(d, d) / r2
            H[3*i:3*i+3, 3*j:3*j+3] = bloque; H[3*j:3*j+3, 3*i:3*i+3] = bloque
            H[3*i:3*i+3, 3*i:3*i+3] -= bloque; H[3*j:3*j+3, 3*j:3*j+3] -= bloque
val, vec = np.linalg.eigh(H)
val, vec = val[6:], vec[:, 6:] # los 6 primeros son traslación y rotación (frecuencia 0)

# fluctuación predicha por residuo
msf_anm = np.full(N_RES, np.nan)
msf_anm[ok] = ((vec**2 / val).reshape(n, 3, -1).sum(axis=1)).sum(axis=1)
B = bfactores_ca(descargar("4AKE"), c_ab[0]["cadena"])
ambos = ok & np.isfinite(B)
r_B = np.corrcoef(B[ambos], msf_anm[ambos])[0, 1]

# traslape de cada modo con el cambio real abierta-cerrada
delta = np.where(np.isfinite(ca_ce_core), ca_ce_core - ca_ab, 0)[ok].ravel()
delta /= np.linalg.norm(delta)
traslape = np.abs(vec[:, :10].T @ delta)
acumulado = np.sqrt(np.cumsum(traslape**2))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})
sombrear_dominios(ax[0])
escala = np.nanmean(B[ambos]) / np.nanmean(msf_anm[ambos])
ax[0].plot(RES, B, color="k", lw=1.2, label="factor B (cristal)")
ax[0].plot(RES, escala * msf_anm, color=ROJO, lw=1.2, ls="--", label="red elástica")
ax[0].set_xlabel("residuo"); ax[0].set_ylabel("B (Å²)"); ax[0].set_xlim(1, N_RES)
ax[0].set_title(f"Flexibilidad: cristal contra resortes · r = {r_B:.2f}"); ax[0].legend(fontsize=7, ncol=2)
ax[0].grid(alpha=0.25)
ax[1].bar(range(1, 11), traslape, color=MORADO, alpha=0.85, label="cada modo")
ax[1].plot(range(1, 11), acumulado, "o-", color="k", ms=4, label="acumulado")
ax[1].set_xlabel("modo (del más lento al más rápido)"); ax[1].set_ylabel("traslape con abierta→cerrada")
ax[1].set_ylim(0, 1); ax[1].set_title("¿Los modos lentos apuntan al cierre?")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()

k = int(np.argmax(traslape[:3]))
print(f"Correlación factores B vs red elástica: r = {r_B:.2f}")
print(f"El modo {k+1} solo tiene un traslape de {traslape[k]:.2f} con el cambio abierta→cerrada;")
print(f"los primeros 3 modos juntos, {acumulado[2]:.2f}  (1 = describirían el cambio completo).")

# animación del modo con más traslape
modo = vec[:, k].reshape(n, 3)
amp = 8.0 / np.linalg.norm(modo, axis=1).max()     # el Cα que más se mueve, 8 Å
signo = np.sign(modo.ravel() @ delta)
marcos = np.array([X + s * signo * amp * modo for s in np.sin(np.linspace(0, 2*np.pi, 24))]) / 10
top_ca = mdt.Topology(); ch = top_ca.add_chain()
for r in RES[ok]:
    top_ca.add_atom("CA", mdt.element.carbon, top_ca.add_residue("GLY", ch, resSeq=int(r)))
mdt.Trajectory(marcos, top_ca).save_pdb("modo_anm.pdb")
v = py3Dmol.view(width=760, height=420)
v.addModelsAsFrames(open("modo_anm.pdb").read(), "pdb")
v.setStyle({}, {"sphere": {"radius": 1.1, "color": C_DOM["CORE"]}})
v.addStyle({"resi": _lista("NMP")}, {"sphere": {"radius": 1.1, "color": C_DOM["NMP"]}})
v.addStyle({"resi": _lista("LID")}, {"sphere": {"radius": 1.1, "color": C_DOM["LID"]}})
v.animate({"loop": "forward", "interval": 70}); v.zoomTo(); v.show()
print(f"Modo {k+1} de la red elástica, exagerado (esferas = Cα).")

Un modelo que sólo conoce la **forma** de la proteína ya reproduce buena parte de su flexibilidad y apunta en la dirección del cambio funcional. La arquitectura de la AdK está "diseñada" para cerrarse.

Este modelo no nos dice cuánta energía cuesta cerrarse, cuánto tiempo tarda, ni qué tan poblado está cada estado. Para eso necesitamos un modelo con física realista, tiempo explícito y el solvente: **dinámica molecular**.

---
# Parte B: Preparar y correr una simulación

Vamos a preparar y simular **1AKE sin el inhibidor**: la forma cerrada, pero sin ligando. Es la misma condición de partida que la simulación de 1 µs que analizaremos la próxima sesión.

Primero comprobamos que la instalación de la celda 0b terminó.

In [ ]:
#@title Verificar la instalación de OpenMM y AmberTools
import shutil
while INSTALACION.poll() is None:
    ultimas = open("instalacion.log").read().strip().splitlines()[-1:]
    print("…todavía instalando:", ultimas[0][:90] if ultimas else ""); time.sleep(15)
log = open("instalacion.log").read()
if "INSTALACION_TERMINADA" not in log:
    print(log[-2000:]); raise RuntimeError("La instalación falló; revisen el log de arriba.")

import openmm as mm
from openmm import unit
from openmm.app import (PDBFile, Modeller, ForceField, PME, HBonds, AllBonds,
                        HAngles, Simulation, DCDReporter, StateDataReporter, element)
import mdtraj.utils.unit as _mdt_unit
_mdt_unit.openmm_unit = unit
print("OpenMM version:", mm.version.version, "\n")
for i in range(mm.Platform.getNumPlatforms()):
    p = mm.Platform.getPlatform(i)
    print(f"  [{i}] {p.getName():10s}  velocidad relativa = {p.getSpeed()}")
nombres = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print("\nCUDA disponible " if "CUDA" in nombres else
      "\n*** NO hay CUDA. Cambien el entorno a GPU T4 y ejecuten todo otra vez. ***")
print("cpptraj:", shutil.which("cpptraj") or "NO ENCONTRADO")

---
## Paso 1: La estructura de partida

Del PDB de 1AKE nos quedamos con la cadena A (el cristal tiene dos copias de la proteína) y quitamos todo lo que no sea proteína: el inhibidor Ap5A y las aguas del cristal.

> ### Pregunta
> Empezar desde la estructura cerrada o abierta es una decisión clave. ¿Qué esperarían ver si empezáramos desde 4AKE?

In [ ]:
#@title Elegir y limpiar la estructura de partida
ESTRUCTURA = "1AKE"   #@param ["1AKE", "4AKE"]
CADENA     = "A"      #@param ["A", "B"]
from collections import Counter

crudo = PDBFile(descargar(ESTRUCTURA))
mod = Modeller(crudo.topology, crudo.positions)
mod.delete([ch for ch in mod.topology.chains() if ch.id != CADENA])
fuera = [r for r in mod.topology.residues() if r.name not in AA]
quitados = Counter(r.name for r in fuera)
mod.delete(fuera)
with open("01_modelo_elegido.pdb", "w") as f:
    PDBFile.writeFile(mod.topology, mod.positions, f)

n_res = mod.topology.getNumResidues()
print(f"{ESTRUCTURA}, cadena {CADENA}: {n_res} residuos, {mod.topology.getNumAtoms()} átomos")
print(f"Quitamos: {dict(quitados)}")
ver("01_modelo_elegido.pdb", ESTILO_DOMINIOS)

In [ ]:
#@title VER: superficie por dominio
#@markdown La hendidura entre LID, NMP y CORE es donde se unen el ATP y el AMP.
v = py3Dmol.view(width=800, height=520)
v.addModel(open("01_modelo_elegido.pdb").read(), "pdb")
for dom in ("CORE", "NMP", "LID"):
    v.addSurface(py3Dmol.VDW, {"opacity": 0.95, "color": C_DOM[dom]}, {"resi": _lista(dom)})
v.zoomTo(); v.show()

---
## Paso 2: Protonar

Las estructuras del PDB pueden están incompletas: sin hidrógenos, a veces con cadenas laterales truncadas o residuos faltantes. Primero revisamos qué le falta.

Los rayos X casi no ven los hidrógenos, así que una estructura cristalográfica no los trae.

> ### Pregunta
> Algunas estructuras sí traen hidrógenos. Entonces:
>
> **1. ¿Deberíamos quitarlos? ¿Por qué sí o por qué no?**
>
> **2. ¿Cómo sabemos dónde poner los hidrógenos?**

In [ ]:
#@title Comprobar integridad y añadir protones
PH                  = 7.0             #@param {type:"number"}
CAMPO_FUERZA_SOLUTO = "amber14-all.xml"   #@param ["amber14-all.xml", "amber19-all.xml", "amber99sbildn.xml", "charmm36.xml", "charmm36_2024.xml"]
MODELO_AGUA         = "tip3p"             #@param ["tip3p", "opc3", "spce", "opc", "tip4pew"]

#@markdown - `PH`: pH al que se asignarán los estados de protonación
#@markdown - `CAMPO_FUERZA_SOLUTO`: Campo de fuerza a utilizar con los solutos (proteína)
#@markdown - `MODELO_AGUA`: Modelo de agua a utilizar (los parámetros de los iones tambien están aquí.)

_GEOMETRIA = {"tip3p": "tip3p", "tip3pfb": "tip3p", "opc3": "tip3p", "spce": "spce",
              "tip4pew": "tip4pew", "tip4pfb": "tip4pew", "opc": "tip4pew",
              "tip4p2005": "tip4pew", "tip5p": "tip5p"}

_AMBER_MOD = ["tip3p", "tip3pfb", "opc3", "spce", "tip4pew", "tip4pfb", "opc"]
_AGUAS = {
    "amber14-all.xml":   {w: f"amber14/{w}.xml" for w in _AMBER_MOD},
    "amber19-all.xml":   {w: f"amber19/{w}.xml" for w in _AMBER_MOD},
    "amber99sbildn.xml": {w: f"{w}.xml" for w in _AMBER_MOD + ["tip5p"]},
    "charmm36.xml":      {w: f"charmm36/{w}.xml" for w in
                          ["spce", "tip4pew", "tip5p", "tip4p2005"]} | {"tip3p": "charmm36/water.xml"},
    "charmm36_2024.xml": {w: f"charmm36_2024/{w}.xml" for w in
                          ["spce", "tip4pew", "tip5p", "tip4p2005"]} | {"tip3p": "charmm36_2024/water.xml"},
}

if MODELO_AGUA not in _AGUAS[CAMPO_FUERZA_SOLUTO]:
    raise ValueError(
        f"'{MODELO_AGUA}' no existe para {CAMPO_FUERZA_SOLUTO}.\n"
        f"Disponibles: {sorted(_AGUAS[CAMPO_FUERZA_SOLUTO])}")

XML_AGUA  = _AGUAS[CAMPO_FUERZA_SOLUTO][MODELO_AGUA]
GEOM_AGUA = _GEOMETRIA[MODELO_AGUA]
print(f"Campo de fuerza: {CAMPO_FUERZA_SOLUTO}")
print(f"Modelo de agua: {XML_AGUA}")

from pdbfixer import PDBFixer
n_H_orig = sum(1 for a in mod.topology.atoms() if a.element == element.hydrogen)

fixer = PDBFixer(filename="01_modelo_elegido.pdb")

fixer.findMissingResidues()
cad = list(fixer.topology.chains())
internos = {k: v for k, v in fixer.missingResidues.items()
            if k[1] != 0 and k[1] != len(list(cad[k[0]].residues()))}
n_extremos = len(fixer.missingResidues) - len(internos)
fixer.missingResidues = internos

fixer.findNonstandardResidues(); fixer.replaceNonstandardResidues()
fixer.removeHeterogens(keepWater=False)
fixer.findMissingAtoms()
n_pesados = sum(len(v) for v in fixer.missingAtoms.values())

print("="*60)
print(f"Huecos internos a rellenar    : {len(fixer.missingResidues)}")
print(f"Extremos desordenados omitidos: {n_extremos}")
print(f"Átomos pesados faltantes      : {n_pesados}")
print(f"Residuos no estándar          : {fixer.nonstandardResidues}")
print("="*60)

fixer.addMissingAtoms()

sin_h = Modeller(fixer.topology, fixer.positions)
sin_h.delete([a for a in sin_h.topology.atoms() if a.element == element.hydrogen])
n_sin_H = sin_h.topology.getNumAtoms()
with open("02a_sin_hidrogenos.pdb", "w") as f:
    PDBFile.writeFile(sin_h.topology, sin_h.positions, f)

forcefield = ForceField(CAMPO_FUERZA_SOLUTO, XML_AGUA)
mod_h = Modeller(sin_h.topology, sin_h.positions)
mod_h.addHydrogens(forcefield, pH=PH)
with open("02b_protonado.pdb", "w") as f:
    PDBFile.writeFile(mod_h.topology, mod_h.positions, f)

n_H = sum(1 for a in mod_h.topology.atoms() if a.element == element.hydrogen)
print(f"\nHidrógenos en el archivo del PDB : {n_H_orig}")
print(f"Átomos pesados                  : {n_sin_H}")
print(f"Hidrógenos puestos a pH {PH}     : {n_H}")
print(f"Total                           : {mod_h.topology.getNumAtoms()}")
print(f"\nLos hidrógenos son el {100*n_H/mod_h.topology.getNumAtoms():.1f}% del número de átomos.")

In [ ]:
#@title VER: antes y después de protonar
v = py3Dmol.view(width=880, height=400, viewergrid=(1, 2))

v.addModel(open("02a_sin_hidrogenos.pdb").read(), "pdb", {"keepH": True}, viewer=(0, 0))
v.setStyle({}, {"stick": {"radius": 0.13, "color": GRIS}}, viewer=(0, 0))

v.addModel(open("02b_protonado.pdb").read(), "pdb", {"keepH": True}, viewer=(0, 1))
v.setStyle({}, {"stick": {"radius": 0.13, "color": GRIS}}, viewer=(0, 1))
v.addStyle({"elem": "H"}, {"sphere": {"radius": 0.25, "color": "#ef4444"}}, viewer=(0, 1))

v.zoomTo()
v.show()

print("Izquierda: sólo átomos pesados.   Derecha: con hidrógenos (rojo).")

---
## Paso 3 - El campo de fuerza

Un **campo de fuerza** es una expresión analítica de la energía potencial en función de las coordenadas. No es una aproximación a la mecánica cuántica: es una función empírica, parametrizada para reproducir observables concretos.

$$U = \underbrace{\sum_{\text{enlaces}} k_b (r-r_0)^2 + \sum_{\text{ángulos}} k_\theta (\theta-\theta_0)^2 + \sum_{\text{diedros}} \frac{V_n}{2}[1+\cos(n\phi-\gamma)]}_{\text{términos enlazados}} + \underbrace{\sum_{i<j} 4\epsilon_{ij}\left[\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12} - \left(\frac{\sigma_{ij}}{r_{ij}}\right)^{6}\right] + \sum_{i<j} \frac{q_i q_j}{4\pi\epsilon_0 r_{ij}}}_{\text{términos no enlazados}}$$

`amber14-all.xml` incluye **ff14SB** para proteínas.

> ### Pregunta
> La AdK une ATP y AMP, que tienen varios fosfatos con carga negativa.
> **¿Qué carga neta esperan para la proteína, y de qué signo?**

In [ ]:
#@title Asignar campo de fuerza y calcular la carga
mod2 = Modeller(mod_h.topology, mod_h.positions)
sistema_vacio = forcefield.createSystem(mod2.topology)
nbf_ = [f for f in sistema_vacio.getForces() if isinstance(f, mm.NonbondedForce)][0]

q_total = sum(nbf_.getParticleParameters(i)[0].value_in_unit(unit.elementary_charge)
              for i in range(sistema_vacio.getNumParticles()))
print("="*50)
for ch in mod2.topology.chains():
    q = sum(nbf_.getParticleParameters(a.index)[0].value_in_unit(unit.elementary_charge)
            for r in ch.residues() for a in r.atoms())
    res = list(ch.residues())
    tipo = "RNA" if res[0].name.strip() in ("A","U","G","C") else "proteína"
    print(f"  Carga de la cadena {ch.id} ({tipo:8s}) : {q:+7.2f} e")
print("-"*50)
print(f"  CARGA TOTAL DE LA PROTEÍNA      : {q_total:+7.2f} e")
print("="*50)

La carga neta es la suma de Arg y Lys (positivas), Asp y Glu (negativas), las histidinas que queden protonadas (+1; su pKa ≈ 6 está muy cerca del pH fisiológico, así que dependen del pH que elegimos y de su entorno) y los extremos de la cadena. Antes de simular hay que neutralizar la carga, porque la electrostática con condiciones periódicas (PME) supone un sistema neutro.

In [ ]:
#@title ¿Dónde están las cargas?
#@markdown Azul = Arg y Lys (positivas). Rojo = Asp y Glu (negativas). Verde = His (neutras o +1, según su protonación).

#@markdown Fíjense en la hendidura entre LID, NMP y CORE: ahí se acomodan los fosfatos del ATP y el AMP.
v = py3Dmol.view(width=780, height=440)
v.addModel(open("02b_protonado.pdb").read(), "pdb")
v.setStyle({}, {"cartoon": {"color": "#e2e8f0"}})
v.addStyle({"resn": ["ARG", "LYS"]}, {"stick": {"radius": 0.22, "color": "#2563eb"}})
v.addStyle({"resn": ["ASP", "GLU"]}, {"stick": {"radius": 0.22, "color": "#dc2626"}})
v.addStyle({"resn": ["HIS", "HID", "HIE", "HIP"]}, {"stick": {"radius": 0.22, "color": "#16a34a"}})
v.zoomTo(); v.show()

> ### Nota sobre los parámetros
>
> **ff14SB** es un estándar para proteínas plegadas en AMBER; **ff19SB** (diseñado para usarse con el agua OPC) y **CHARMM36m** corrigen algunos de sus sesgos, sobre todo en regiones flexibles o desordenadas.
>
> La simulación de 1 µs que analizaremos el jueves se corrió con **CHARMM27**, otro campo de fuerza y otra duración. Al comparar su corrida con esa, tengan en mente que no sólo cambia el tiempo.
>
> `MODELO_AGUA` les deja probar combinaciones. No hay una respuesta correcta, hay que evaluar cual funciona mejor para cada sistema y cada pregunta.

---
## Paso 4: Solvatar el sistema

Metemos la proteína en agua explícita con **condiciones de contorno periódicas (PBC)**: la caja se repite infinitamente en las tres direcciones.

**Convención de imagen mínima.** Cada átomo interactúa sólo con la copia *más cercana* de cada otro átomo. Para que eso sea válido, la molécula **nunca** debe ver su propia imagen periódica.

**¿Y por qué una caja octaédrica en vez de un cubo?** Porque una biomolécula es más o menos globular, y un cubo desperdicia las esquinas: para garantizar la misma distancia mínima entre imágenes en todas las direcciones, un cubo necesita mucha más agua. Un **octaedro truncado** llena el espacio con ~77 % del volumen de un cubo equivalente, una **dodecaedro**, con ~71 %.

Ese ahorro es directamente tiempo de cómputo, porque el costo lo domina el solvente.

> ### Pregunta
> Con 1.0 nm de margen: **¿cuántas moléculas de agua creen que caben?**
> ¿Decenas, cientos, miles? ¿Y cuánta agua nos ahorra el octaedro?

In [ ]:
#@title Solvatar y neutralizar
FORMA_CAJA = "octahedron"   #@param ["octahedron", "dodecahedron", "cube"]
PADDING_NM = 1            #@param {type:"number"}
SAL_MOLAR  = 0.15           #@param {type:"number"}
CATION     = "Na+"          #@param ["Na+", "K+", "Li+", "Rb+", "Cs+"]
ANION      = "Cl-"          #@param ["Cl-", "Br-", "F-", "I-"]

#@markdown - `FORMA_CAJA`: Forma de la caja
#@markdown - `PADDING_NM`: Distancia del borde de la caja a cualquier punto en la proteína, en nm
#@markdown - `SAL_MOLAR`: Concentración de sal **después** de neutralizar la carga
#@markdown - `CATION` Catión a elegir
#@markdown - `ANION`: Anión a elegir.

from collections import Counter
_t0 = time.time()

mod2 = Modeller(mod_h.topology, mod_h.positions)
mod2.addSolvent(forcefield, model=GEOM_AGUA, boxShape=FORMA_CAJA,
                padding=PADDING_NM*unit.nanometer,
                positiveIon=CATION, negativeIon=ANION,
                ionicStrength=SAL_MOLAR*unit.molar, neutralize=True)

top, pos = mod2.topology, mod2.positions
NOMBRES_ION = {"NA","CL","K","LI","RB","CS","BR","F","I"}
n_aguas = sum(1 for r in top.residues() if r.name in ("HOH","WAT"))
conteo_iones = Counter(r.name for r in top.residues() if r.name in NOMBRES_ION)

vec = np.array(top.getPeriodicBoxVectors().value_in_unit(unit.nanometer))
volumen = abs(np.linalg.det(vec))
anchos = np.array([volumen/np.linalg.norm(np.cross(vec[1], vec[2])),
                   volumen/np.linalg.norm(np.cross(vec[2], vec[0])),
                   volumen/np.linalg.norm(np.cross(vec[0], vec[1]))])

idx_soluto = [a.index for a in top.atoms()
              if a.residue.name not in ("HOH","WAT") and a.residue.name not in NOMBRES_ION]
xyz = np.array(pos.value_in_unit(unit.nanometer))[idx_soluto]
diametro = 2*np.linalg.norm(xyz - xyz.mean(axis=0), axis=1).max()
vol_cubo = (diametro + 2*PADDING_NM)**3

sis_tmp = forcefield.createSystem(top, nonbondedMethod=PME)
nb_tmp = [f for f in sis_tmp.getForces() if isinstance(f, mm.NonbondedForce)][0]
q_fin = sum(nb_tmp.getParticleParameters(i)[0].value_in_unit(unit.elementary_charge)
            for i in range(sis_tmp.getNumParticles()))

print("="*64)
print(f"  Átomos totales del sistema    : {top.getNumAtoms():8d}")
print(f"  Moléculas de agua             : {n_aguas:8d}")
print(f"  Iones                         : {dict(conteo_iones)}")
print(f"  CARGA TOTAL                   : {q_fin:+8.2f} e   <-- debe ser ~0")
print("-"*64)
print(f"  Forma de caja                 : {FORMA_CAJA}")
print(f"  Volumen                       : {volumen:8.1f} nm³")
print(f"  Un cubo equivalente sería     : {vol_cubo:8.1f} nm³ "
      f"({100*volumen/vol_cubo:.0f} % del cubo)")
print(f"  Agua ahorrada frente al cubo  : ~{100*(1-volumen/vol_cubo):.0f} %")
print("="*64)
print(f"\n(solvatación en {time.time()-_t0:.0f} s)")

with open("03_solvatado.pdb", "w") as f:
    PDBFile.writeFile(top, pos, f)

v = py3Dmol.view(width=800, height=480)
v.addModel(open("03_solvatado.pdb").read(), "pdb")

# Render water molecules as tiny spheres since they act as single atoms here
v.setStyle({"resn": ["HOH","WAT"]}, {"sphere": {"radius": 0.15, "color": C_AGUA, "opacity": 1}})

for _sel, _est in ESTILO_DOMINIOS: v.addStyle(_sel, _est)

# Separate ions into positive and negative groups based on common naming conventions
pos_ions = [ion for ion in conteo_iones.keys() if any(p in ion.upper() for p in ['NA', 'K', 'MG', 'CA', 'ZN', '+'])]
neg_ions = [ion for ion in conteo_iones.keys() if any(n in ion.upper() for n in ['CL', 'CLA', 'BR', '-'])]

# Apply distinct colors (Blue for positive, Red for negative)
if pos_ions:
    v.addStyle({"resn": pos_ions}, {"sphere": {"radius": 0.8, "color": "#3b82f6"}}) # Blue
if neg_ions:
    v.addStyle({"resn": neg_ions}, {"sphere": {"radius": 0.8, "color": "#ef4444"}}) # Red

v.zoomTo(); v.show()
print("Azul = iones positivos | Rojo = iones negativos")

---
## Paso 5: Minimización de energía

**La minimización no es dinámica molecular.** No hay temperatura, no hay velocidades, no hay tiempo. Es un algoritmo de optimización que baja por el gradiente de la superficie de energía potencial hasta el mínimo local más cercano.

¿Para qué sirve? Después de protonar y solvatar (incluso desde la estructura experimental) hay contactos incorrectos, hidrógenos recién colocados muy cerca, aguas extremadamente ordenadas. Si arrancáramos la dinámica en ese punto, esas fuerzas mandarían átomos a velocidades absurdas y la integración explotaría.

Minimizar remueve estos contactos antes de empezar.

> ### Pregunta
> **¿La energía potencial va a subir o a bajar, y en qué orden de magnitud?**
> **¿Cuánto se va a mover la estructura: décimas de Å, o nanómetros?**

In [ ]:
#@title Construir el sistema y minimizar
CORTE_NM      = 1.0 #@param {type:"number"}
RESTRICCIONES = "HBonds"
AGUA_RIGIDA   = True
TEMPERATURA_K = 300        #@param {type:"number"}
FRICCION_PS   = 1.0
PASO_FS       = 2.0        #@param {type:"number"}
MAX_ITER_MIN  = 5000       #@param {type:"integer"}
PLATAFORMA    = "CUDA"

#@markdown - `CORTE_NM`: *Cutoff* para las interacciones no enlazadas, en nm
#@markdown - `TEMPERATURA_K`: Temperatura a la que se realizará la simulación, en K (se usa después de la minimización).
#@markdown - `PASO_FS`: Paso de integración, en fs.
#@markdown - `MAX_ITER_MIN`: Número máximos de ciclos de minimización.


_CONSTR = {"HBonds": HBonds, "AllBonds": AllBonds, "HAngles": HAngles, "None": None}

sistema = forcefield.createSystem(top, nonbondedMethod=PME,
                                  nonbondedCutoff=CORTE_NM*unit.nanometer,
                                  constraints=_CONSTR[RESTRICCIONES],
                                  rigidWater=AGUA_RIGIDA)
integrador = mm.LangevinMiddleIntegrator(TEMPERATURA_K*unit.kelvin,
                                         FRICCION_PS/unit.picosecond,
                                         PASO_FS*unit.femtoseconds)
sim = Simulation(top, sistema, integrador, mm.Platform.getPlatformByName(PLATAFORMA))
sim.context.setPositions(pos)

pos_ini = np.array(sim.context.getState(getPositions=True)
                   .getPositions().value_in_unit(unit.nanometer))
E_ini = sim.context.getState(getEnergy=True).getPotentialEnergy()

_t0 = time.time(); sim.minimizeEnergy(maxIterations=MAX_ITER_MIN); _dt = time.time()-_t0

E_fin = sim.context.getState(getEnergy=True).getPotentialEnergy()
pos_fin = np.array(sim.context.getState(getPositions=True)
                   .getPositions().value_in_unit(unit.nanometer))
desplaz = np.linalg.norm(pos_fin - pos_ini, axis=1)

print("="*60)
print(f"  Energía potencial ANTES   : {E_ini.value_in_unit(unit.kilojoule_per_mole):14.1f} kJ/mol")
print(f"  Energía potencial DESPUÉS : {E_fin.value_in_unit(unit.kilojoule_per_mole):14.1f} kJ/mol")
print(f"  Diferencia                : {(E_fin-E_ini).value_in_unit(unit.kilojoule_per_mole):14.1f} kJ/mol")
print("-"*60)
print(f"  Desplazamiento medio del soluto : {10*desplaz[idx_soluto].mean():.3f} Å")
print(f"  Desplazamiento máximo (todo)    : {10*desplaz.max():.3f} Å")
print(f"  (tiempo: {_dt:.0f} s)")
print("="*60)
print('''
La energía cae en decenas de miles de kJ/mol, pero los átomos apenas se movieron. Toda esa energía estaba concentrada en unos pocos contactos malos. La estructura no cambió: sólo se quitaron las espinas.
''')
with open("04_minimizado.pdb", "w") as f:
    PDBFile.writeFile(top, sim.context.getState(getPositions=True).getPositions(), f)

In [ ]:
#@title VER cuánto se movió el soluto al minimizar
#@markdown Gris = antes de minimizar, Color = después. Deberían verse casi encimados
import mdtraj as mdt
_m = mdt.load("04_minimizado.pdb")
_sel = _m.topology.select("not water and not resname NA CL K LI RB CS BR F I")
_m.atom_slice(_sel).save_pdb("04b_min_soluto.pdb")

v = py3Dmol.view(width=900, height=440)
v.addModel(open("02b_protonado.pdb").read(), "pdb")
v.setStyle({"model": 0}, {"cartoon": {"color": "#94a3b8"}})
v.addModel(open("04b_min_soluto.pdb").read(), "pdb")
v.setStyle({"model": 1}, {"cartoon": {"color": C_PROT}})
v.zoomTo(); v.show()
print("\n")

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(10*desplaz[idx_soluto], bins=60, color=C_PROT, alpha=0.85)
ax.set_xlabel("desplazamiento al minimizar (Å)")
ax.set_ylabel("número de átomos"); ax.grid(alpha=0.25)
ax.set_title("La mayoría de los átomos se mueve poco")
plt.tight_layout(); plt.show()

---
## Paso 6: Equilibración NVT (calentar)

Después de minimizar, todos los átomos tienen **velocidad cero**: el sistema está a 0 K.

> ### Pregunta
> **¿De dónde salen las velocidades?**

Se asignan al azar desde la distribución de Maxwell–Boltzmann a la temperatura objetivo. La condición inicial de una simulación de DM es aleatoria. Esta es la razón por la que **réplicas independientes con distintas semillas** son la forma correcta de estimar incertidumbre.

**Restricciones de posición:** durante el equilibrado restringimos los átomos pesados de los solutos con resortes, porque el agua todavía no se acomodó. Si dejamos todo libre desde el primer paso, la estructura se relaja hacia lo que el campo de fuerza prefiera, *mientras* el solvente sigue desordenado, y perdemos la estructura experimental antes de empezar a medir.

In [ ]:
#@title Restricciones posicionales + equilibración NVT
PS_NVT           = 25     #@param {type:"number"}
K_RESTRICCION    = 500.0  #@param {type:"number"}
SEMILLA          = 42   #@param {type:"integer"}
REPORTAR_CADA_PS = 0.5    #@param {type:"number"}

#@markdown - `PS_NVT`: Cuántos *ps* de calentamiento realizar
#@markdown - `K_RESTRICCION`: Fuerza de las restricciones sobre los átomos pesados (kJ/mol/A^2)
#@markdown - `SEMILLA`: Semilla para asignar velocidades iniciales (la misma semilla reproduce los mismos parámetros)
#@markdown - `REPORTAR_CADA_PS`: Intervalo de tiempo en el que se guardan los datos

import sys
pasos_nvt = int(PS_NVT*1000/PASO_FS)
rep_cada = max(1, int(REPORTAR_CADA_PS*1000/PASO_FS))
set_soluto = set(idx_soluto)

for i in reversed(range(sistema.getNumForces())):
    if isinstance(sistema.getForce(i), mm.CustomExternalForce):
        sistema.removeForce(i)

restr = mm.CustomExternalForce("k*periodicdistance(x, y, z, x0, y0, z0)^2")
restr.addGlobalParameter("k", K_RESTRICCION*unit.kilojoules_per_mole/unit.nanometer**2)
for p in ("x0","y0","z0"): restr.addPerParticleParameter(p)
ref = sim.context.getState(getPositions=True).getPositions()
n_restr = 0
for atom in top.atoms():
    if atom.index in set_soluto and atom.element != element.hydrogen:
        restr.addParticle(atom.index, ref[atom.index].value_in_unit(unit.nanometer))
        n_restr += 1
sistema.addForce(restr)
sim.context.reinitialize(preserveState=True)
print(f"Restricciones aplicadas a {n_restr} átomos pesados del soluto.\n")

integrador.setRandomNumberSeed(SEMILLA)
sim.context.setVelocitiesToTemperature(TEMPERATURA_K*unit.kelvin, SEMILLA)
print(f"Velocidades desde Maxwell-Boltzmann a {TEMPERATURA_K} K (semilla = {SEMILLA})")
print("Cambien la semilla y obtendrán otra trayectoria. Eso es una réplica.\n")

sim.reporters.clear()
sim.reporters.append(StateDataReporter("nvt.csv", rep_cada, step=True, time=True,
                                       potentialEnergy=True, kineticEnergy=True,
                                       totalEnergy=True, temperature=True, volume=True))
sim.reporters.append(StateDataReporter(sys.stdout, max(1, pasos_nvt//8), step=True,
                                       time=True, temperature=True,
                                       speed=True, separator="  |  "))
_t0 = time.time(); sim.step(pasos_nvt)
print(f"\nNVT terminado en {time.time()-_t0:.0f} s")
with open("05_nvt.pdb", "w") as f:
    PDBFile.writeFile(top, sim.context.getState(getPositions=True,
                      enforcePeriodicBox=True).getPositions(), f)

In [ ]:
#@title GRAFICAR: temperatura y energías durante el NVT
import pandas as pd
d = pd.read_csv("nvt.csv")
col = lambda k: d[[c for c in d.columns if k in c][0]]
t = col("Time")

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].plot(t, col("Temperature"), lw=1.1, color=C_PROT)
ax[0].axhline(TEMPERATURA_K, ls="--", c=C_ACENTO, lw=1.4, label=f"{TEMPERATURA_K} K objetivo")
ax[0].set_ylabel("Temperatura (K)"); ax[0].set_title("Temperatura"); ax[0].legend(fontsize=8)

ax[1].plot(t, col("Potential"), lw=1.1, color="#7c3aed")
ax[1].set_ylabel("E potencial (kJ/mol)"); ax[1].set_title("Energía potencial")

ax[2].plot(t, col("Kinetic"), lw=1.1, color="#dc2626", label="cinética")
ax[2].plot(t, col("Total"), lw=1.1, color="#0f766e", label="total")
ax[2].set_ylabel("Energía (kJ/mol)"); ax[2].set_title("Cinética y total"); ax[2].legend(fontsize=8)

for a in ax: a.set_xlabel("tiempo (ps)"); a.grid(alpha=0.25)
plt.suptitle("Equilibración NVT: # particulas, volumen y temperatura constante")
plt.tight_layout(); plt.show()

T = col("Temperature")
print(f"T promedio en la 2a mitad: {T[len(T)//2:].mean():.1f} ± {T[len(T)//2:].std():.1f} K")

La temperatura llega al objetivo en pocos *ps* y luego fluctúa alrededor. Esas fluctuaciones son normales, en un sistema finito la temperatura
instantánea fluctúa como $\frac{1}{\sqrt{N}}$.

La energía cinética sube (estamos calentando) mientras la potencial se acomoda. En NVT la energía total **NO** se conserva: el termostato permite un intercambio de energía a propósito.

---
## Paso 7: Equilibración NPT

Hasta ahora el volumen estuvo fijo. Pero nosotros pusimos las aguas con una densidad dada por las dimensiones de la caja. No tiene por qué ser la correcta a 300 K y 1 bar de presión.

Activamos un **barostato**: el volumen de la caja puede cambiar, y el sistema encuentra por sí mismo su densidad de equilibrio.

In [ ]:
#@title Equilibración NPT con barostato
PS_NPT         = 100    #@param {type:"number"}
PRESION_BAR    = 1.0   #@param {type:"number"}
FREC_BAROSTATO = 25

#@markdown - `PS_NPT`: Duración de la equilibración NPT, en ps
#@markdown - `PRESION_BAR`: Presión objetivo para el barostato, en bar


pasos_npt = int(PS_NPT*1000/PASO_FS)

for i in reversed(range(sistema.getNumForces())):
    if isinstance(sistema.getForce(i), mm.MonteCarloBarostat):
        sistema.removeForce(i)

sistema.addForce(mm.MonteCarloBarostat(PRESION_BAR*unit.bar,
                                       TEMPERATURA_K*unit.kelvin, FREC_BAROSTATO))
sim.context.reinitialize(preserveState=True)

sim.reporters.clear()
sim.reporters.append(StateDataReporter("npt.csv", rep_cada, step=True, time=True,
                                       potentialEnergy=True, kineticEnergy=True,
                                       totalEnergy=True, temperature=True,
                                       volume=True, density=True))
sim.reporters.append(StateDataReporter(sys.stdout, max(1, pasos_npt//8), step=True,
                                       time=True, temperature=True, density=True,
                                       speed=True, separator="  |  "))
_t0 = time.time(); sim.step(pasos_npt)
print(f"\nNPT terminado en {time.time()-_t0:.0f} s")
with open("06_npt.pdb", "w") as f:
    PDBFile.writeFile(top, sim.context.getState(getPositions=True,
                      enforcePeriodicBox=True).getPositions(), f)

In [ ]:
#@title GRAFICAR: qué hace el barostato
d = pd.read_csv("npt.csv")
col = lambda k: d[[c for c in d.columns if k in c][0]]
t = col("Time")

fig, ax = plt.subplots(2, 2, figsize=(12, 6.6))
ax[0,0].plot(t, col("Density"), lw=1.1, color=C_PROT)
ax[0,0].axhline(0.997, ls="--", c=C_ACENTO, lw=1.4, label="agua pura, 300 K")
ax[0,0].set_ylabel("densidad (g/cm³)"); ax[0,0].set_title("Densidad"); ax[0,0].legend(fontsize=8)

ax[0,1].plot(t, col("Box Volume"), lw=1.1, color=C_ACENTO)
ax[0,1].set_ylabel("volumen (nm³)"); ax[0,1].set_title("Volumen de la caja")

ax[1,0].plot(t, col("Temperature"), lw=1.1, color="#dc2626")
ax[1,0].axhline(TEMPERATURA_K, ls="--", c=GRIS, lw=1.2)
ax[1,0].set_ylabel("Temperatura (K)"); ax[1,0].set_title("El termostato sigue activo")
ax[1,0].set_ylim(290, 310)

ax[1,1].plot(t, col("Potential"), lw=1.1, color="#7c3aed")
ax[1,1].set_ylabel("E potencial (kJ/mol)"); ax[1,1].set_title("Energía potencial")

for a in ax.ravel(): a.set_xlabel("tiempo (ps)"); a.grid(alpha=0.25)
plt.suptitle("Equilibración NPT: # particulas, presión y temperatura constante")
plt.tight_layout(); plt.show()

rho = col("Density"); m = len(rho)//2
print(f"Densidad promedio (2a mitad): {rho[m:].mean():.4f} ± {rho[m:].std():.4f} g/cm³")
print(f"Volumen promedio  (2a mitad): {col('Box Volume')[m:].mean():.1f} nm³")

---
## Paso 8: Producción

Quitamos las restricciones posicionales y dejamos el sistema libre. Esta es la parte que vamos a analizar.

**Sobre el paso de integración.** El integrador tiene que resolver el movimiento más rápido del sistema: la vibración de estiramiento X–H, con período de unos 10 fs. Eso exigiría ~1 fs.

Pero pusimos `constraints=HBonds`, que fija la longitud de los enlaces a hidrógeno y elimina el problema de los movimientos más rápidos.


> ### Pregunta
> A 2 fs por paso: ¿cuántos pasos son 200 ps? ¿Y 1 µs?
>
> **¿Qué les dice esto sobre lo que están a punto de obtener?**

In [ ]:
#@title Producción: corrida corta
PS_PRODUCCION   = 500   #@param {type:"number"}
GUARDAR_CADA_PS = 10    #@param {type:"number"}

#@markdown - `PS_PRODUCCION`: Duración de la simulación, en ps
#@markdown - `GUARDAR_CADA_PS`: Intervalo para guardar los datos.

#@markdown **DETALLE**: 200 ps es un tiempo muy corto, el sistema sigue equilibrándose. Es sólo un ejercicio para que alcance a terminar durante la clase.
#@markdown El jueves analizaremos una simulación de 1 µs del mismo sistema.

pasos_prod = int(PS_PRODUCCION*1000/PASO_FS)
guardar_cada = max(1, int(GUARDAR_CADA_PS*1000/PASO_FS))

for i in reversed(range(sistema.getNumForces())):
    if isinstance(sistema.getForce(i), mm.CustomExternalForce):
        sistema.removeForce(i)
sim.context.reinitialize(preserveState=True)
print("Restricciones posicionales eliminadas. El soluto está libre.\n")
print(f"{pasos_prod:,} pasos x {PASO_FS} fs = {PS_PRODUCCION} ps")
print(f"Se guardarán {pasos_prod//guardar_cada} cuadros (uno cada {GUARDAR_CADA_PS} ps)")
print(f"Para comparar: 1 µs = {int(1e6*1000/PASO_FS):,} pasos, "
      f"{int(1e6/PS_PRODUCCION)}x más de lo que haremos aquí.\n")

sim.reporters.clear()
sim.reporters.append(DCDReporter("produccion_corta.dcd", guardar_cada))
sim.reporters.append(StateDataReporter("produccion.csv", guardar_cada, step=True,
                                       time=True, potentialEnergy=True, temperature=True,
                                       density=True, volume=True))
sim.reporters.append(StateDataReporter(sys.stdout, max(1, pasos_prod//10), step=True,
                                       time=True, temperature=True,
                                       speed=True, separator="  |  "))
_t0 = time.time(); sim.step(pasos_prod); _dt = time.time()-_t0

print(f"\nProducción terminada en {_dt/60:.1f} min")
vel = (PS_PRODUCCION/1000)/(_dt/86400)
print(f"Rendimiento: {vel:.1f} ns/día")
print(f"A este ritmo, 1 µs tomaría {1000/vel:.0f} días en este Colab.")
print("Ésa es exactamente la razón por la que la trayectoria larga viene pre-calculada.")

with open("07_topologia_final.pdb", "w") as f:
    PDBFile.writeFile(top, sim.context.getState(getPositions=True,
                      enforcePeriodicBox=True).getPositions(), f)

---
## Paso 9: Centrar y reimaginar la trayectoria

En una caja periódica las moléculas no se quedan dentro de la caja. Cuando un átomo sale por un lado, el programa lo reintroduce por el opuesto. Si uno visualiza el archivo crudo, hay moléculas "partidas" en pedazos, o que "saltan" de un lado a otro.

Esto es un artefacto de cómo se guardan las coordenadas.

**Reimaginar** (*imaging*) significa reconstruir cada molécula entera y volver a centrar el soluto. **Alinear** significa quitar la traslación y la rotación globales, que no nos interesan. Es importante realizar este paso antes de cualquier analisis.

In [ ]:
#@title Centrar, reimaginar y alinear
ALINEAR_SOBRE = "name CA"  # alineamos sobre los Cα de la proteína

import subprocess, textwrap, os

SEL_STRIP = ":HOH,WAT,NA,CL,K,LI,RB,CS,BR"

guion = textwrap.dedent(f"""\
    parm 07_topologia_final.pdb
    trajin produccion_corta.dcd

    center :1-{n_res}
    image familiar
    rms first :1-{n_res}@CA

    # version CON disolvente, para ver la caja
    outtraj 08_octaedro_con_agua.pdb pdb
    outtraj 08_trayectoria_agua.xtc

    # fuera disolvente y contraiones
    strip {SEL_STRIP}

    trajout 08_trayectoria_centrada.xtc
    trajout 08_topologia_centrada.pdb pdb onlyframes 1

    go
    quit
    """)

with open("autoimage.in", "w") as f:
    f.write(guion)

r = subprocess.run(["cpptraj", "-i", "autoimage.in"], capture_output=True, text=True)
#print(r.stdout[-2500:])
if r.returncode != 0:
    print(r.stderr[-2500:])
    raise RuntimeError("cpptraj falló — revisen la salida de arriba.")

traj = mdt.load("08_trayectoria_centrada.xtc", top="08_topologia_centrada.pdb")
print(f"\nProcesada: {traj.n_frames} cuadros, {traj.n_atoms} átomos, "
      f"alineada sobre '{ALINEAR_SOBRE}'")

In [ ]:
#@title VER la caja octaédrica
v = py3Dmol.view(width=800, height=520)
v.addModel(open("08_octaedro_con_agua.pdb").read(), "pdb", {"keepH": True})
v.setStyle({"resn": ["HOH", "WAT"]}, {"sphere": {"radius": 0.12, "color": C_AGUA}})
for _sel, _est in ESTILO_DOMINIOS: v.addStyle(_sel, _est)
v.zoomTo(); v.show()
print("Ahora sí se ve el octaedro truncado.")

In [ ]:
#@title VER la simulación, animada **SIN AGUA**
traj = mdt.load("08_trayectoria_centrada.xtc", top="08_topologia_centrada.pdb")
# 1. Downsample the trajectory (e.g., every 10th frame) to avoid freezing the browser
traj_subset = traj

# 2. Save and load via temporary file
temp_file = "temp_traj_view.pdb"
traj_subset.save_pdb(temp_file)

with open(temp_file, "r") as f:
    pdb_string = f.read()

os.remove(temp_file)

# 3. Initialize viewer and load frames
view = py3Dmol.view(width=800, height=600)
view.addModelsAsFrames(pdb_string, "pdb")

# 4. Apply your specific styles across ALL frames ('model': -1)
for _sel, _est in ESTILO_DOMINIOS: view.addStyle(dict(_sel, model=-1), _est)

# 5. Animate and render
view.animate({'loop': 'forward', 'reps': 0, 'interval': 100})
view.zoomTo()
view.show()
print(f"Ésta es su simulación. Cada cuadro son {GUARDAR_CADA_PS} ps de movimiento real.")
print('''
Miren con atención qué dominio se mueve más. ¿Se abre la proteína? Lo cuantificamos en la Parte C.
''')

In [ ]:
#@title VER la simulación, animada **CON AGUA** (con cuidado)
traj = mdt.load("08_trayectoria_agua.xtc", top="08_octaedro_con_agua.pdb")
# 1. Downsample the trajectory (e.g., every 10th frame) to avoid freezing the browser
traj_subset = traj

# 2. Save and load via temporary file
temp_file = "temp_traj_view.pdb"
traj_subset.save_pdb(temp_file)

with open(temp_file, "r") as f:
    pdb_string = f.read()

os.remove(temp_file)

# 3. Initialize viewer and load frames
view = py3Dmol.view(width=800, height=600)
view.addModelsAsFrames(pdb_string, "pdb", {'keepH': True})

# 4. Apply your specific styles across ALL frames ('model': -1)
for _sel, _est in ESTILO_DOMINIOS: view.addStyle(dict(_sel, model=-1), _est)

# 5. Animate and render
view.animate({'loop': 'forward', 'reps': 0, 'interval': 100})
view.zoomTo()
view.show()
print(f"Ésta es su simulación. Cada cuadro son {GUARDAR_CADA_PS} ps de movimiento real.")
print('''
Miren con atención qué dominio se mueve más. ¿Se abre la proteína? Lo cuantificamos en la Parte C.
''')

---
# Parte C: ¿Qué cambió en 500 ps?

Empezamos en la forma **cerrada sin ligando**. En la Parte A vimos que, sin ligando, la AdK suele cristalizar abierta.

> ### Pregunta
> **¿Se abrió la proteína en ese tiempo? ¿Cuánto creen que cambie la distancia LID–CORE?**

In [ ]:
#@title C: Distancias entre dominios y flexibilidad en su corrida
traj = mdt.load("08_trayectoria_centrada.xtc", top="08_topologia_centrada.pdb")
ica = traj.topology.select("name CA")
resseq = np.array([traj.topology.atom(i).residue.resSeq for i in ica])
if len(ica) == N_RES and resseq[0] != 1:
    resseq = RES  # cpptraj puede renumerar; AdK tiene 214
ca_t = np.full((traj.n_frames, N_RES, 3), np.nan)
ca_t[:, resseq - 1] = 10 * traj.xyz[:, ica]
t_ps = np.arange(traj.n_frames) * GUARDAR_CADA_PS

# referencias de la Parte A
_, c_ab_ = cadenas_adk(descargar("4AKE")); _, c_ce_ = cadenas_adk(descargar("1AKE"))
dN_ab, dL_ab = distancias_dominio(c_ab_[0]["ca"]); dN_ce, dL_ce = distancias_dominio(c_ce_[0]["ca"])

dN_t, dL_t = distancias_dominio(ca_t)
al = np.array([alinear(c, ca_t[0], MASK["CORE"])[0] for c in ca_t])
rmsd_t = np.array([rmsd(a, al[0]) for a in al])
rmsf_t = np.sqrt(((al - al.mean(0))**2).sum(-1).mean(0))
B_ce = bfactores_ca(descargar(ESTRUCTURA), CADENA)
rmsf_B = np.sqrt(3 * B_ce / (8 * np.pi**2))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1, 1.2, 1.6]})
ax[0].plot(t_ps, rmsd_t, "o-", ms=3, color=C_PROT)
ax[0].set_xlabel("tiempo (ps)"); ax[0].set_ylabel("RMSD Cα (Å)"); ax[0].set_title("RMSD (alineando el CORE)")
for d_t, ab_, ce_, nom, c in ((dL_t, dL_ab, dL_ce, "LID–CORE", C_DOM["LID"]),
                              (dN_t, dN_ab, dN_ce, "NMP–CORE", C_DOM["NMP"])):
    ax[1].plot(t_ps, d_t, lw=1.5, color=c, label=f"{nom} (su corrida)")
    ax[1].axhline(ab_, ls="--", color=c, lw=1); ax[1].axhline(ce_, ls=":", color=c, lw=1)
ax[1].set_xlabel("tiempo (ps)"); ax[1].set_ylabel("distancia (Å)")
ax[1].set_title("--- abierta (4AKE)   ··· cerrada (1AKE)"); ax[1].legend(fontsize=7)
sombrear_dominios(ax[2])
ax[2].plot(RES, rmsf_t, color="k", lw=1.2, label=f"RMSF de su corrida ({PS_PRODUCCION} ps)")
ax[2].plot(RES, rmsf_B, color=ROJO, lw=1.1, ls="--", label="RMSF estimado de los factores B")
ax[2].set_xlabel("residuo"); ax[2].set_ylabel("RMSF (Å)"); ax[2].set_xlim(1, N_RES)
ax[2].set_title("¿Qué se movió?"); ax[2].legend(fontsize=7, ncol=2)
for a in ax: a.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"LID–CORE: empezó en {dL_t[0]:.1f} Å y terminó en {dL_t[-1]:.1f} Å. "
      f"Abierta (4AKE) es {dL_ab:.1f} Å: le faltan {dL_ab - dL_t[-1]:.1f} Å.")
print(f"NMP–CORE: empezó en {dN_t[0]:.1f} Å y terminó en {dN_t[-1]:.1f} Å "
      f"(abierta {dN_ab:.1f} Å).")
print(f"""
En {PS_PRODUCCION:g} ps la proteína vibra, los loops se mueven y las cadenas laterales se
reacomodan, pero los dominios no se abren.

El jueves: la MISMA simulación (1AKE sin ligando), de 1 µs.""")

---
## Guardar el trabajo

El jueves pueden comparar su dinámica contra la trayectoria de 1 µs. Descarguen el archivo y súbanlo al cuaderno del jueves si quieren compararlas.

In [ ]:
#@title Empaquetar y descargar resultados
NOMBRE = f"resultados_{ESTRUCTURA}_semilla{SEMILLA}"
!tar -czf {NOMBRE}.tar.gz 08_trayectoria_centrada.xtc 08_topologia_centrada.pdb 08_octaedro_con_agua.pdb 08_trayectoria_agua.xtc\
        produccion_corta.dcd 07_topologia_final.pdb nvt.csv npt.csv produccion.csv 2>/dev/null
!ls -lh {NOMBRE}.tar.gz

try:
    from google.colab import files
    files.download(f"{NOMBRE}.tar.gz")
except Exception as e:
    print("Descarga manual desde el panel de archivos:", e)

---
# Ejercicio para el jueves

En clase simularon la AdK partiendo de la forma **cerrada sin ligando** (1AKE). Ahora repitan la simulación partiendo de la forma **abierta** (4AKE).

**Qué hacer**:

1. En el **Paso 1** cambien `ESTRUCTURA` a `"4AKE"`.
2. Volver a correr la preparación y simulación.
3. Descarguen el archivo `resultados_4AKE_semilla….tar.gz` de la última celda. Guarden también el de 1AKE que bajaron en clase: el jueves vamos a cargar los dos.

**Qué contestar** (con las gráficas de la Parte C de cada corrida):

1. Partiendo de la forma abierta, ¿se cerró algo en la simulación? Comparen la distancia LID–CORE al inicio y al final con las líneas de referencia.
2. ¿Cuál de las dos simulaciones se movió más? ¿Por qué podría moverse más una proteína cerrada **sin** el ligando que la mantiene cerrada?
3. ¿Sus dos simulaciones se parecen más entre sí, o cada una a su estructura inicial? ¿Qué nos dice eso de lo que se puede concluir con este tiempo de simulación?
4. El jueves veremos 1 µs partiendo de 1AKE sin ligando. ¿Se abrirá? Si se abre, ¿qué fracción del tiempo creen que pasará abierta?

---

## Para llevarse

**Sobre la dinámica conformacional**

1. **Una estructura es una fotografía; la proteína es un ensamble.** El PDB ya contiene a la AdK abierta, cerrada y en estados intermedios.
2. **El cambio funcional de la AdK es de dominios rígidos con bisagras:** pocos residuos cambian su geometría y el resto se mueve en bloque.
3. **La forma determina buena parte del movimiento.**
4. **AlphaFold predice una estructura, no un ensamble**, y no dice cuánto tiempo pasa la proteína en cada estado. Ni el cristal tampoco.

**Sobre la simulación**

5. **La estructura de partida es una decisión clave.** Empezamos en 1AKE sin ligando, y esa elección se propaga a todo.
6. **Los hidrógenos no vienen del experimento.** Son casi la mitad de los átomos y los modelamos nosotros, a un pH que elegimos.
7. **La forma de la caja es importante.** El octaedro truncado da la misma distancia mínima entre imágenes con ~23 % menos agua que un cubo.
8. La minimización **no es** dinámica: baja mucha energía moviendo muy poco.
9. Las velocidades iniciales son **aleatorias**. Por eso se corren réplicas.
10. "Equilibrado" no es propiedad del sistema, sino **de un observable**: la densidad converge en decenas de ps.
11. **500 ps no alcanzan para ver un cambio conformacional funcional.**

**Jueves:** la misma simulación, de 1 µs, y cómo se analiza e interpreta.